In [1]:
from datasets import load_dataset

ds = load_dataset("sander-wood/irishman")

/Users/andrewmcknight/code/ceol-gpt/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating validation split: 100%|██████████| 2162/2162 [00:00<00:00, 253518.00 examples/s]


In [2]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['abc notation', 'control code'],
        num_rows: 214122
    })
    validation: Dataset({
        features: ['abc notation', 'control code'],
        num_rows: 2162
    })
})


In [3]:
ds["train"][0]

{'abc notation': 'X:1\nL:1/8\nM:4/4\nK:Emin\n|: E2 EF E2 EF | DEFG AFDF | E2 EF E2 B2 |1 efe^d e2 e2 :|2 efe^d e3 B |: e2 ef g2 fe | \n defg afdf |1 e2 ef g2 fe | efe^d e3 B :|2 g2 bg f2 af | efe^d e2 e2 ||',
 'control code': 'S:2\nB:5\nE:5\nB:6\n'}

In [5]:
for abc in ds["train"][0:4]['abc notation']:
    print(abc)
    print("-" * 80)

X:1
L:1/8
M:4/4
K:Emin
|: E2 EF E2 EF | DEFG AFDF | E2 EF E2 B2 |1 efe^d e2 e2 :|2 efe^d e3 B |: e2 ef g2 fe | 
 defg afdf |1 e2 ef g2 fe | efe^d e3 B :|2 g2 bg f2 af | efe^d e2 e2 ||
--------------------------------------------------------------------------------
X:2
L:1/4
M:3/4
K:C
 G | E3/2 E/ E | G2 G | c2 c | G2 G | G2 c/>c/ | B2 d | d3 | c2 G | e2 e | e d c | B2 A | A2 A | 
 B2 B | d2 d | d3 | c2 G | e2 e | e d c | B2 A | A2 A | B2 B | d2 d | d3 | c2 |]
--------------------------------------------------------------------------------
X:3
L:1/8
Q:1/4=100
M:2/4
K:Emin
 E | ABcA | E2 EF | G2 GB | (d/c/B/A/) GB | AGAB | E2 E=f | edcB | A2 A :: B | c>Bce | d>cde | 
 c>Bce | (d/c/B/A/) GB | c>Bce | dcd=f | edcB | A2 A :|
--------------------------------------------------------------------------------
X:4
L:1/8
M:6/8
K:A
|: EFG ||"A" A2 A A2 A |"F#m" A2 A A2 A |"B7" B2 A G2 F |"E7" E6 |"A" A2 A A3 |"D" A2 A A3 | 
"A" c2 A"E7" B2 G |"^A fine" A3 :|"A7" A2 A ||"D" F6- | F3 A2 A |"A" E6- | 

In [9]:
import pandas as pd

df = ds["train"].to_pandas()
df = df.drop(columns=["control code"]).rename(columns={"abc notation": "abc"})
df.head()

,abc
0,X:1\nL:1/8\nM:4/4\nK:Emin\n|: E2 EF E2 EF | DE...
1,X:2\nL:1/4\nM:3/4\nK:C\n G | E3/2 E/ E | G2 G ...
2,X:3\nL:1/8\nQ:1/4=100\nM:2/4\nK:Emin\n E | ABc...
3,"X:4\nL:1/8\nM:6/8\nK:A\n|: EFG ||""A"" A2 A A2 A..."
4,X:5\nL:1/8\nM:6/8\nK:G\n (G/A/B).D D>ED | DcB ...


In [12]:
import re

def extract_header(text: str, tag: str, default: str | None = None) -> str | None:
    """Return the ABC header value for a given tag (e.g., K, M, L)."""
    if not isinstance(text, str) or not isinstance(tag, str) or not tag:
        return default

    pattern = rf"(?m)^\s*{re.escape(tag)}\s*:\s*(.+?)\s*$"
    match = re.search(pattern, text)
    return match.group(1).strip() if match else default

df["key"] = df["abc"].apply(lambda x: extract_header(x, "K"))
df["time_signature"] = df["abc"].apply(lambda x: extract_header(x, "M"))
df["note_length"] = df["abc"].apply(lambda x: extract_header(x, "L"))

df[["key", "time_signature", "note_length"]].head()

,key,time_signature,note_length
0,Emin,4/4,1/8
1,C,3/4,1/4
2,Emin,2/4,1/8
3,A,6/8,1/8
4,G,6/8,1/8


In [16]:
df["key"].value_counts()

key
G        59848
D        49458
C        21516
A        15614
F        12326
         ...  
Cb           1
Dloc         1
Eblyd        1
C#phr        1
C#           1
Name: count, Length: 62, dtype: int64

In [17]:
df["time_signature"].value_counts()

time_signature
4/4      70726
6/8      46046
2/4      30972
3/4      25790
2/2      25328
         ...  
63/84        1
26/8         1
12/6         1
6/9          1
3/6          1
Name: count, Length: 80, dtype: int64

In [18]:
df["note_length"].value_counts()

note_length
1/8     180684
1/4      18539
1/16     14899
Name: count, dtype: int64